# 02 ResNet 分类与临床融合
读取原尺寸四层缓存，用固定的ResNet50提取影像特征，训练后续特征投影、病灶分支、注意力和分类头，再训练临床及融合模型。结果仍保存到 `results/resnet`。

本次回到“原尺寸输入＋ImageNet标准化＋整个主干固定”的设置，取消第6轮起微调最后一个残差块。结果表优先显示漏判、误报数量和对应错误率，准确率及AUC继续保留供参考。


## 本次恢复的版本与关注指标
2026-11-26恢复原尺寸输入、ImageNet标准化、整个ResNet主干固定的训练设置。它此前的融合结果为AUC 0.803241、准确率75.1724%、敏感度0.777778、特异度0.718750。按开发集81例阳性、64例阴性换算，漏判18例、误报18例。

最早提供的融合结果是漏判14例、误报21例，准确率75.8621%；那一版总错误少1例，漏判更少，但误报更多。当前选择主干固定、标准化这一版作为后续数据复核的工作基础，是为了同时关注两类错误，并不表示它在所有指标上最优。历史结果来自不同输入或训练流程，不是严格的单变量比较。

本次取消末端残差块微调，整个主干及其BatchNorm参数、统计量始终固定。只训练256维影像投影、64维病灶分支、四层门控注意力和分类头；学习率仍为5e-4。每批16位患者，每折完整训练30轮，再恢复该折留出AUC最高的权重。最终模型仍按五折最佳轮数中位数训练。

原尺寸缓存、输入标准化、增强、损失函数、AdamW、权重衰减、梯度裁剪、患者分组和随机种子均保持原样。已有原尺寸缓存可以继续使用；如果另外剔除或重标了患者，需要先完成相应的清单及缓存更新。

以下将“漏判”记为FN：阳性判为阴性；将“误报”记为FP：阴性判为阳性。FNR=FN/全部阳性，FPR=FP/全部阴性。阈值仍由开发集Youden指数确定，相当于在候选阈值中最小化FNR+FPR，兼顾两类错误率。没有另设漏判成本或敏感度目标，也不强行降低阈值。阈值的变化可能使一类错误减少、另一类增加，不能保证两者同时下降。

开发折参与了选轮数和阈值选择，开发成绩可能偏乐观；时间测试由06完成，使用开发阶段确定的阈值，不在测试集重新挑阈值。这里恢复的是训练代码，并未恢复历史权重或重新产生上述成绩。

参考：[分类概率与判定阈值](https://scikit-learn.org/stable/modules/classification_threshold.html)。


## 调整前的02结果
2026-11-22记录，来自此前已完成的02运行。以下为开发队列145例患者的折外结果，阳性81例、阴性64例。

| 模型 | 准确率 | AUC | 阈值 | 敏感度 | 特异度 | Brier |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| ResNet | 62.7586% | 0.685378 | 0.525308 | 0.481481 | 0.812500 | 0.244820 |
| Clinical5 | 71.7241% | 0.774884 | 0.516418 | 0.666667 | 0.781250 | 0.219752 |
| ResNet_Clinical5 | 72.4138% | 0.774498 | 0.505578 | 0.716049 | 0.734375 | 0.210473 |

影像模型五折平均AUC为0.719551。各折AUC和选定训练轮数如下：

| 折 | AUC | 轮数 |
| --- | ---: | ---: |
| 1 | 0.833333 | 21 |
| 2 | 0.572115 | 8 |
| 3 | 0.826923 | 6 |
| 4 | 0.649038 | 1 |
| 5 | 0.716346 | 15 |

当时采用前三轮预热，之后微调最后一组残差块，主干学习率1e-5；训练轮数由训练患者内部另划20%选择，再重新训练该折模型。这与本次恢复的选轮数流程不同。表格只记录调整前的成绩，本次运行结果会显示在后面的训练单元，并更新原结果目录中的CSV。

### 关闭提前停止前的这次结果
本次运行使用固定ResNet主干、按留出折AUC选最佳权重，仍有“连续8轮AUC未提高就停止”的设置。开发队列仍为145例。

| 模型 | 准确率 | 整体折外AUC | 五折平均AUC | 五折AUC标准差 | Brier |
| --- | ---: | ---: | ---: | ---: | ---: |
| ResNet | 66.2069% | 0.687114 | 0.753356 | 0.100141 | 0.234728 |
| Clinical5 | 71.7241% | 0.774884 | 0.786897 | 0.064567 | 0.219752 |
| ResNet_Clinical5 | 73.1034% | 0.757137 | 0.805279 | 0.063152 | 0.219186 |

影像阈值0.542197，敏感度0.469136、特异度0.906250；融合阈值0.510383，敏感度0.716049、特异度0.750000。最终影像模型训练3轮，来自五折最佳轮数的中位数。这些是关闭提前停止前的历史成绩。

### 加入标准化、保留原尺寸之前的结果
以下是此前一次完整训练30轮的成绩：使用224×224缓存，固定ResNet主干，尚未加入ImageNet均值/标准差处理。开发队列145例。

| 模型 | 准确率 | 整体折外AUC | 五折平均AUC | 五折AUC标准差 | Brier |
| --- | ---: | ---: | ---: | ---: | ---: |
| ResNet | 69.6552% | 0.739680 | 0.777413 | 0.086811 | 0.221414 |
| Clinical5 | 71.7241% | 0.774884 | 0.786897 | 0.064567 | 0.219752 |
| ResNet_Clinical5 | 73.1034% | 0.793981 | 0.818910 | 0.076124 | 0.211946 |

影像阈值0.399987，敏感度0.679012、特异度0.718750；融合阈值0.524701，敏感度0.617284、特异度0.875000。

| 折 | AUC | 最佳轮数 |
| --- | ---: | ---: |
| 1 | 0.877451 | 28 |
| 2 | 0.697115 | 26 |
| 3 | 0.846154 | 27 |
| 4 | 0.653846 | 3 |
| 5 | 0.812500 | 9 |

最佳轮数中位数为26。以上是调整输入之前的记录，不是当前代码已训练出的结果。

### 末端微调之前：原尺寸输入、标准化、整个主干固定
2026-11-22记录，来自本次修改前提供的开发队列结果，样本量145例。沿用原尺寸缓存和ImageNet输入标准化，每折完整训练30轮后恢复最佳权重，当时整个ResNet主干仍固定。

| 模型 | 准确率 | 整体折外AUC | 阈值 | 敏感度 | 特异度 | Brier | 五折平均AUC | 五折AUC标准差 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Clinical5 | 71.7241% | 0.774884 | 0.516418 | 0.666667 | 0.781250 | 0.219752 | 0.786897 | 0.064567 |
| ResNet_Clinical5 | 75.1724% | 0.803241 | 0.500224 | 0.777778 | 0.718750 | 0.204710 | 0.816101 | 0.084866 |

该轮尚未提供影像单独模型的指标和各折结果，因此这里只记录已提供的两组数据，不沿用更早的影像AUC代替。后来尝试末端微调，仅收到“提升不大”的反馈，没有完整数值，不为这次微调补写指标。

### 按漏判与误报回看已提供的融合结果
以下均为历史开发集145例（阳性81、阴性64），数量由已报告的敏感度、特异度换算；每行使用其当时开发阈值。不是独立时间测试，也不能据此判定某一版纯ResNet最优。

| 版本 | AUC | 准确率 | 漏判FN | 误报FP | 总错误 | 漏判率FNR | 误报率FPR |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| 最早提供的ResNet50＋Clinical5 | 0.808835 | 75.8621% | 14 | 21 | 35 | 17.2840% | 32.8125% |
| 224输入、主干固定、完整30轮 | 0.793981 | 73.1034% | 31 | 8 | 39 | 38.2716% | 12.5000% |
| 原尺寸＋标准化、主干固定（本次恢复） | 0.803241 | 75.1724% | 18 | 18 | 36 | 22.2222% | 28.1250% |
| Clinical5临床对照 | 0.774884 | 71.7241% | 27 | 14 | 41 | 33.3333% | 21.8750% |

本次恢复版相较224版漏判减少13例、误报增加10例；相较最早版漏判增加4例、误报减少3例。没有一版在这三组比较中同时取得最少漏判和最少误报。重训或数据清理后的结果需要重新计算，不能沿用本表数量。


## 路径、数据与训练参数
先确认打印的缓存尺寸与原图一致。当前按A800设置每批16位患者，训练使用混合精度；本机未测量512输入的实际显存占用。00、02、04、06读取同一份缓存即可，无需新增配置文件。


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
# 从项目根目录打开时，临床分析脚本位于 1117 子目录。
import sys
if str(ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))
from clinical_association import CLINICAL5, CLINICAL7, add_clinical7
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


In [ ]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.ndim != 5 or images.shape[1:3] != (4, 3) or masks.shape != images.shape or min(images.shape[-2:]) < 32:
        raise ValueError("分类输入应为[N,4,3,H,W]，影像与mask尺寸须一致，H和W至少为32。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


In [ ]:
OUTPUT = RESULTS / "resnet"
TRAIN = {
    "batch_size": 16,
    "max_epochs": 30,
    "head_lr": 5e-4,
}
MODEL = {"image_dim": 256, "mask_dim": 64, "attention_dim": 128,
         "use_mask": True, "use_attention": True, "normalize_input": True}
RUN_TRAINING = False
RUN_CLINICAL = True
RESTART = True

if RUN_TRAINING:
    cases, images, masks = read_classification_data()
    print("缓存中每层实际尺寸：", images.shape[-2:])
    print("影像进入ResNet前使用ImageNet均值/标准差；mask保持0/1。")
else:
    cases = read_cases()
display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))


## 网络结构与预测
同一套ResNet50逐层编码三序列影像，病灶编码器处理三个标签通道，拼接后对四层加权汇总，输出一个患者级概率。自适应池化将空间特征汇总为固定维度，输入保留512×512时，分类头无需改变。

训练全程固定主干参数及其BatchNorm统计量。调用 `model.train()` 时，后续模块进入训练模式，主干保持预测模式；不再开放最后一个残差块。

输入标准化仍在模型内部，训练、预测和Grad-CAM走同一段处理。04、06按PT权重中的 `model_args` 读取，无需重复标准化；网络结构和权重格式不变。

训练载入ImageNet初始权重，已有缓存时直接使用，否则torchvision会下载一次。测试只读取保存的实验权重。


In [ ]:
import random
import torch
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("计算设备：", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class GatedAttention(nn.Module):
    def __init__(self, features, hidden=128):
        super().__init__()
        self.value = nn.Linear(features, hidden)
        self.gate = nn.Linear(features, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.score(torch.tanh(self.value(x)) * torch.sigmoid(self.gate(x)))


class MaskEncoder(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, features),
            nn.LayerNorm(features), nn.GELU(), nn.Dropout(.3))

    def forward(self, x):
        return self.net(x.float())


class ResNetClassifier(nn.Module):
    def __init__(self, image_dim=256, mask_dim=64, attention_dim=128,
                 use_mask=True, use_attention=True, pretrained=False, normalize_input=False):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.image_features.requires_grad_(False)
        self.image_features.eval()
        self.normalize_input = normalize_input
        self.register_buffer("input_mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1), persistent=False)
        self.register_buffer("input_std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1), persistent=False)
        self.image_projection = nn.Sequential(nn.Linear(2048, image_dim), nn.LayerNorm(image_dim), nn.GELU(), nn.Dropout(.3))
        self.mask_features = MaskEncoder(mask_dim) if use_mask else None
        width = image_dim + (mask_dim if use_mask else 0)
        self.attention = GatedAttention(width, attention_dim) if use_attention else None
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Dropout(.4), nn.Linear(width, 1))

    def train(self, mode=True):
        super().train(mode)
        # 后续模块照常训练，固定主干及其BatchNorm统计量。
        self.image_features.eval()
        return self

    def forward(self, images, masks):
        b, slices, channels, h, w = images.shape
        image_input = images.reshape(b * slices, channels, h, w)
        if self.normalize_input:
            image_input = (image_input - self.input_mean) / self.input_std
        features = self.image_projection(self.image_features(image_input).flatten(1))
        features = features.reshape(b, slices, -1)
        if self.mask_features is not None:
            lesion = self.mask_features(masks.reshape(b * slices, 3, h, w)).reshape(b, slices, -1)
            features = torch.cat([features, lesion], dim=-1)
        if self.attention is None:
            pooled = features.mean(dim=1)
        else:
            weights = self.attention(features).softmax(dim=1)
            pooled = (features * weights).sum(dim=1)
        return self.head(pooled).squeeze(-1)


def sequence_input(images, sequence="all"):
    if sequence == "all":
        return images
    channel = SEQUENCES.index(sequence)
    return images[:, :, [channel, channel, channel]]


@torch.no_grad()
def predict(model, images, masks, batch_size=16):
    model.eval()
    probabilities = []
    for start in range(0, len(images), batch_size):
        x = torch.as_tensor(images[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        m = torch.as_tensor(masks[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(x, m)
        probabilities.append(logits.float().sigmoid().cpu().numpy())
    return np.concatenate(probabilities)


def load_classifier(file):
    saved = torch.load(file, map_location="cpu", weights_only=True)
    model = ResNetClassifier(**saved["model_args"], pretrained=False).to(DEVICE)
    model.load_state_dict(saved["state_dict"])
    return model.eval(), saved


## 数据增强与评价指标
影像和标签采用相同的旋转、平移和缩放，四层保持一致；标签使用最近邻插值。亮度和噪声仅加到影像上。

每折用最佳轮权重生成预测，再合并全部开发患者的折外概率。CSV中的 `auc` 对应整体折外AUC，`mean_fold_auc`、`std_fold_auc` 为五折AUC的均值和标准差。

本次在指标表中加入 `fn`、`fp`、`fnr`、`fpr`：分别是漏判人数、误报人数、漏判率、误报率。FNR=1−敏感度，FPR=1−特异度，两种率分别以阳性和阴性人数为分母。数量与比例一起看，避免患者数变化后仅看错误人数。

开发阈值仍按Youden指数选取，不修改损失或阈值算法。时间测试沿用保存的阈值；降低阈值本身不会提高AUC，也不能保证同时减少FN和FP。


In [ ]:
def augment_batch(images: torch.Tensor, masks: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    out_images = images.clone()
    out_masks = masks.clone()
    B, S = images.shape[:2]
    for b in range(B):
        angle = random.uniform(-6.0, 6.0)
        translate = (random.uniform(-6.0, 6.0), random.uniform(-6.0, 6.0))
        scale = random.uniform(0.96, 1.04)
        for s in range(S):
            out_images[b, s] = affine(
                images[b, s],
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            out_masks[b, s] = affine(
                masks[b, s].float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).byte()

    jitter = 0.9 + 0.2 * torch.rand(B, 1, 1, 1, 1, device=out_images.device)
    out_images = out_images * jitter
    if random.random() < 0.5:
        out_images = out_images + torch.randn_like(out_images) * 0.008
    out_images = out_images.clamp(0.0, 1.0)
    return out_images, out_masks


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss


def choose_threshold(labels, probabilities):
    fpr, tpr, thresholds = roc_curve(labels, probabilities)
    finite = np.isfinite(thresholds)
    return float(thresholds[finite][np.argmax((tpr - fpr)[finite])])


def classification_metrics(labels, probabilities, threshold):
    labels, probabilities = np.asarray(labels, int), np.asarray(probabilities, float)
    if not np.isfinite(threshold):
        raise ValueError("分类阈值必须是开发队列确定的有限数值。")
    if set(labels) != {0, 1} or not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("指标需要两类结局，以及0～1之间的有效概率。")
    tn, fp, fn, tp = confusion_matrix(labels, probabilities >= threshold, labels=[0, 1]).ravel()
    return {"n": len(labels), "fn": int(fn), "fp": int(fp),
            "fnr": float(fn / (tp + fn)), "fpr": float(fp / (tn + fp)),
            "auc": float(roc_auc_score(labels, probabilities)), "threshold": float(threshold),
            "accuracy": float((tp + tn) / len(labels)), "sensitivity": float(tp / (tp + fn)),
            "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(labels, probabilities))}


## 五折训练和最终模型
开发患者沿用清单中的五折编号，每折用其余四折训练。每轮在留出患者上计算AUC，出现更高AUC时复制权重。每折完整训练30轮，全程固定主干；训练结束后恢复AUC最高的权重，生成折外预测并保存 `fold_1.pt`～`fold_5.pt`。AUC相同时保留较早轮次。

五折结束后，按最佳轮数中位数重新初始化并训练最终模型 `model.pt`，同样全程固定主干。时间测试患者不参与选择轮数或阈值。

默认 `RUN_TRAINING=False`、`RUN_CLINICAL=True`，直接复用已有的 `development_oof.csv` 训练临床及融合模型。需要重训影像模型时，将 `RUN_TRAINING` 改为 `True`；`RESTART=True` 会覆盖同名影像训练结果。

训练开始会打印可训练参数量及主干固定状态。若覆盖旧模型时中断，保持 `RESTART=True` 重新运行，避免混入旧折结果。训练后先看FN、FP及其错误率，再结合AUC和各折表现判断。


In [ ]:
def fit_network(images, masks, labels, train_index, model_args, settings, epochs, seed, valid_index=None):
    set_seed(seed)
    model = ResNetClassifier(**model_args, pretrained=True).to(DEVICE)
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    print(f"可训练参数：{sum(parameter.numel() for parameter in trainable):,}；ResNet主干固定", flush=True)
    optimizer = torch.optim.AdamW(trainable, lr=settings["head_lr"], weight_decay=1e-4)
    y = labels[train_index]
    pos_weight = torch.tensor([max(float((y == 0).sum()) / max(int((y == 1).sum()), 1), .5)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_auc, best_epoch = -1.0, 1
    best_state = None
    for epoch in range(1, epochs + 1):
        model.train()
        shuffled = np.asarray(train_index)[torch.randperm(len(train_index)).numpy()]
        total_loss = 0.0
        for start in range(0, len(shuffled), settings["batch_size"]):
            index = shuffled[start:start + settings["batch_size"]]
            x, m = augment_batch(torch.from_numpy(images[index]), torch.from_numpy(masks[index]))
            yb = torch.as_tensor(labels[index], dtype=torch.float32, device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = criterion(model(x.to(DEVICE), m.to(DEVICE)), yb)
            if not torch.isfinite(loss):
                raise ValueError("损失出现无效数值，请检查输入或降低学习率。")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(index)
        message = f"epoch {epoch:02d}/{epochs}, loss={total_loss / len(train_index):.4f}"
        if valid_index is not None:
            probability = predict(model, images[valid_index], masks[valid_index], settings["batch_size"])
            auc = float(roc_auc_score(labels[valid_index], probability))
            if auc > best_auc:
                best_auc, best_epoch = auc, epoch
                # 独立复制最佳轮权重，后续训练不会改动这份快照。
                best_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}
            message += f", val_auc={auc:.4f}, best_epoch={best_epoch}"
        print(message, flush=True)
    if valid_index is not None:
        model.load_state_dict(best_state)
        print(f"恢复最佳权重：epoch={best_epoch}, val_auc={best_auc:.4f}", flush=True)
    return model, best_epoch if valid_index is not None else epochs


def run_experiment(cases, images, masks, folder, settings, model_args, sequence="all", restart=False):
    folder.mkdir(parents=True, exist_ok=True)
    dev_index = np.flatnonzero(cases.split.eq("development"))
    dev = cases.iloc[dev_index].reset_index(drop=True)
    x, m = sequence_input(images[dev_index], sequence), masks[dev_index]
    y, folds = dev.label.to_numpy(dtype=int), dev.development_fold.to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5}:
        raise ValueError("开发队列需有1～5的固定折号。")
    oof, chosen_epochs, fold_rows, trained = np.zeros(len(dev), np.float32), [], [], False
    for fold in range(1, 6):
        train_index, valid_index = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
        weight_file, prediction_file = folder / f"fold_{fold}.pt", folder / f"fold_{fold}_oof.csv"
        if not restart and weight_file.exists() and prediction_file.exists():
            saved = torch.load(weight_file, map_location="cpu", weights_only=True)
            previous = pd.read_csv(prediction_file, dtype={"mr": str}).set_index("mr")
            if not previous.index.is_unique or set(previous.index) != set(dev.mr.iloc[valid_index]):
                raise ValueError("已有折结果的患者不同；重训请设置RESTART=True。")
            previous = previous.loc[dev.mr.iloc[valid_index]]
            if not np.array_equal(previous.label, y[valid_index]):
                raise ValueError("已有折结果的标签不同；重训请设置RESTART=True。")
            oof[valid_index], epochs = previous.probability.to_numpy(), int(saved["epochs"])
            print(f"fold {fold}：读取已完成结果", flush=True)
        else:
            print(f"fold {fold}：训练{len(train_index)}人，留出{len(valid_index)}人；完整训练{settings['max_epochs']}轮后恢复留出AUC最高的权重", flush=True)
            model, epochs = fit_network(x, m, y, train_index, model_args, settings,
                                        settings["max_epochs"], SEED + 100 + fold, valid_index=valid_index)
            oof[valid_index] = predict(model, x[valid_index], m[valid_index], settings["batch_size"])
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "model_args": model_args,
                        "sequence": sequence, "input_size": list(x.shape[-2:]), "epochs": epochs, "settings": settings,
                        "train_mrs": dev.mr.iloc[train_index].tolist()}, weight_file)
            frame = dev.iloc[valid_index][["mr", "label", "development_fold"]].copy()
            frame["probability"] = oof[valid_index]
            frame.to_csv(prediction_file, index=False)
            del model
            trained = True
        chosen_epochs.append(epochs)
        fold_rows.append({"fold": fold, "auc": float(roc_auc_score(y[valid_index], oof[valid_index])), "epochs": epochs})
    threshold = choose_threshold(y, oof)
    final_file = folder / "model.pt"
    if restart or trained or not final_file.exists():
        final_epochs = max(1, int(np.median(chosen_epochs)))
        print(f"五折最佳轮数：{chosen_epochs}；最终模型按中位数训练{final_epochs}轮", flush=True)
        final, _ = fit_network(x, m, y, np.arange(len(dev)), model_args, settings, final_epochs, SEED + 1000)
        torch.save({"state_dict": {k: v.cpu() for k, v in final.state_dict().items()}, "model_args": model_args,
                    "sequence": sequence, "input_size": list(x.shape[-2:]), "threshold": threshold, "epochs": final_epochs, "settings": settings,
                    "train_mrs": dev.mr.tolist()}, final_file)
        del final
    frame = dev[["mr", "label", "development_fold"]].copy()
    frame["probability"] = oof
    frame.to_csv(folder / "development_oof.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(folder / "fold_metrics.csv", index=False)
    fold_aucs = [row["auc"] for row in fold_rows]
    metrics = {**classification_metrics(y, oof, threshold),
               "mean_fold_auc": float(np.mean(fold_aucs)), "std_fold_auc": float(np.std(fold_aucs))}
    pd.DataFrame([metrics]).to_csv(folder / "development_metrics.csv", index=False)
    return pd.DataFrame([metrics])


In [ ]:
if RUN_TRAINING:
    display(run_experiment(cases, images, masks, OUTPUT, TRAIN, MODEL, restart=RESTART))
else:
    print("复用已有影像折外预测；需要重训影像模型时，将 RUN_TRAINING 改为 True。")


## 临床对照与融合
Clinical5 使用年龄、受教育年限、NIHSS、同型半胱氨酸和尿酸；Clinical7 在此基础上加入 Fazekas 评分（0～3 级）和糖尿病（有=1、无=0）。两组各自与同一份 ResNet 折外概率做晚期融合，分别得到 `ResNet_Clinical5` 和 `ResNet_Clinical7`。MoCA ≤22 定义结局；所有输入缺失值在训练折内填补。

若患者清单已有新增两列，直接使用。旧清单按 MR 从配置中的 Excel 补取这两项。关联分析见同目录的 `clinical_association.py`。

融合沿用一轮折外预测再做逻辑回归的方式。它不是完整嵌套的融合评估，因此融合的开发集指标用于探索，最终效果以独立时间测试为准。02 重新训练后，也要重跑这一节，使融合模型对应当前影像模型。


In [ ]:
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


def clinical_search():
    pipeline = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                         ("scale", StandardScaler()), ("lr", LogisticRegression(solver="liblinear", max_iter=3000))])
    return GridSearchCV(pipeline, {"lr__C": [.003, .01, .03, .1, .3, 1.]}, scoring="roc_auc",
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1, error_score="raise")


def fit_clinical_models(cases, resnet_folder):
    dev = cases.query("split == 'development'").reset_index(drop=True)
    oof_path = resnet_folder / "development_oof.csv"
    if not oof_path.is_file():
        raise FileNotFoundError(f"缺少影像折外预测：{oof_path}。请放入当前 ResNet 的预测文件。")
    image_oof = pd.read_csv(oof_path, dtype={"mr": str}).set_index("mr")
    if not image_oof.index.is_unique or set(image_oof.index) != set(dev.mr):
        raise ValueError("影像OOF与开发患者不一致。")
    image_oof = image_oof.loc[dev.mr]
    if not np.array_equal(image_oof.label, dev.label) or not np.array_equal(image_oof.development_fold, dev.development_fold):
        raise ValueError("影像 OOF 的结局或折号与患者清单不一致。")
    image_prob = image_oof.probability.to_numpy(dtype=float)
    if not np.isfinite(image_prob).all() or (image_prob < 0).any() or (image_prob > 1).any():
        raise ValueError("影像 OOF 的预测概率应在 0～1 之间。")
    y, folds = dev.label.to_numpy(dtype=int), dev.development_fold.to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5} or any(set(y[folds == fold]) != {0, 1} for fold in range(1, 6)):
        raise ValueError("开发清单应有五个固定折，每折都须包含两类结局。")
    dev = add_clinical7(dev, path_from_config("excel_path"), cfg)
    if (dev[CLINICAL7[len(CLINICAL5):]].nunique() < 2).any():
        raise ValueError("开发队列的 Fazekas 和糖尿病各须至少有两种非缺失取值。")
    x5, x7 = (dev[columns].to_numpy(dtype=float) for columns in (CLINICAL5, CLINICAL7))
    inputs = {
        "Clinical5": x5,
        "ResNet_Clinical5": np.column_stack([image_prob, x5]),
        "Clinical7": x7,
        "ResNet_Clinical7": np.column_stack([image_prob, x7]),
    }
    models, rows, tables = {}, [], []
    for name, values in inputs.items():
        probabilities = np.zeros(len(dev), dtype=float)
        fold_aucs = []
        for fold in range(1, 6):
            valid = folds == fold
            search = clinical_search().fit(values[~valid], y[~valid])
            probabilities[valid] = search.predict_proba(values[valid])[:, 1]
            fold_aucs.append(float(roc_auc_score(y[valid], probabilities[valid])))
        threshold = choose_threshold(y, probabilities)
        models[name] = {"model": clinical_search().fit(values, y).best_estimator_, "threshold": threshold}
        rows.append({"model": name, **classification_metrics(y, probabilities, threshold),
                     "mean_fold_auc": float(np.mean(fold_aucs)), "std_fold_auc": float(np.std(fold_aucs))})
        table = dev[["mr", "label", "development_fold"]].copy()
        table["model"], table["probability"] = name, probabilities
        tables.append(table)
    joblib.dump(models, resnet_folder / "clinical_models.joblib")
    pd.DataFrame(rows).to_csv(resnet_folder / "clinical_development_metrics.csv", index=False)
    pd.concat(tables).to_csv(resnet_folder / "clinical_development_oof.csv", index=False)
    return pd.DataFrame(rows)


In [ ]:
if RUN_CLINICAL:
    display(fit_clinical_models(cases, OUTPUT))
else:
    print("准备好对应的影像折外预测后，将 RUN_CLINICAL 改为 True 即可训练临床对照和融合。")


## 主要输出
`model.pt` 是最终影像模型，`clinical_models.joblib` 是临床及融合模型。`development_oof.csv` 保存患者级折外预测，`development_metrics.csv`、`clinical_development_metrics.csv` 增加FN、FP、FNR、FPR。`fold_1.pt`～`fold_5.pt` 用于04的解释。

结果目录仍是 `results/resnet`。临床结果包含 Clinical5、Clinical7 及各自的 ResNet 融合。06可读取这四个模型做时间测试，比较 AUC、准确率和两类错误。

03仍保留此前的训练策略。后续开展当前02的消融时，再同步其输入处理、训练范围和选轮数流程。
